In [0]:
mapping_land_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "bronze/state_elections/state=hamburg/"
    "Landesliste_F_Feldbezeichner.csv"
)

df_mapping_land_raw = (
    spark.read
    .option("header", False)
    .option("sep", ",")
    .csv(mapping_land_path)
)

df_mapping_land_raw.select(
    *df_mapping_land_raw.columns[:10]
).show(15, truncate=False)

+----------------------+--------------------------------------------------------------------------------------+---------+-----------------------+--------------------+-----------------------------+--------------------------+-----------+--------------------------+------------------------------------+
|_c0                   |_c1                                                                                   |_c2      |_c3                    |_c4                 |_c5                          |_c6                       |_c7        |_c8                       |_c9                                 |
+----------------------+--------------------------------------------------------------------------------------+---------+-----------------------+--------------------+-----------------------------+--------------------------+-----------+--------------------------+------------------------------------+
|Statistikamt Nord     |Zuordnung der Parteien und Kandidierenden zu den Feldbezeichnern in der Down

In [0]:
from pyspark.sql import functions as F

mapping_cells = (
    df_mapping_land_raw
    .select(
        F.posexplode(
            F.array(*[F.col(c) for c in df_mapping_land_raw.columns])
        ).alias("column_position", "value")
    )
    .filter(
        F.col("value").rlike(r"^F\d+$")
    )
)

mapping_cells.show(100, truncate=False)

+---------------+-----+
|column_position|value|
+---------------+-----+
|19             |F1   |
|83             |F2   |
|147            |F3   |
|211            |F4   |
|275            |F5   |
|325            |F6   |
|353            |F7   |
|382            |F8   |
|410            |F9   |
|424            |F10  |
|438            |F11  |
|454            |F12  |
|465            |F13  |
|478            |F14  |
|484            |F15  |
|502            |F16  |
+---------------+-----+



In [0]:
start = 15
end = 30

df_mapping_land_raw.select(
    *df_mapping_land_raw.columns[start:end]
).show(15, truncate=False)

+-----------------------+--------------------------+------------------------+--------------------+------------------+---------------------+-------------------+---------------------+-----------------------------+------------------+----------------------+---------------------------+-----------------+-------------------------+-----------------------+
|_c15                   |_c16                      |_c17                    |_c18                |_c19              |_c20                 |_c21               |_c22                 |_c23                         |_c24              |_c25                  |_c26                       |_c27             |_c28                     |_c29                   |
+-----------------------+--------------------------+------------------------+--------------------+------------------+---------------------+-------------------+---------------------+-----------------------------+------------------+----------------------+---------------------------+-----------------+-

In [0]:
from pyspark.sql import functions as F

cells = (
    df_mapping_land_raw
    .select(
        F.posexplode(
            F.array(*[F.col(c) for c in df_mapping_land_raw.columns])
        ).alias("column_position", "value")
    )
)

f_fields = (
    cells
    .filter(
        F.col("value").rlike(r"^F\d+(_[HOP])?$")
    )
)

f_fields.show(100, truncate=False)

+---------------+-----+
|column_position|value|
+---------------+-----+
|19             |F1   |
|20             |F1_H |
|22             |F1_P |
|83             |F2   |
|84             |F2_H |
|86             |F2_P |
|147            |F3   |
|148            |F3_H |
|150            |F3_P |
|211            |F4   |
|212            |F4_H |
|214            |F4_P |
|275            |F5   |
|276            |F5_H |
|278            |F5_P |
|325            |F6   |
|326            |F6_H |
|328            |F6_P |
|353            |F7   |
|354            |F7_H |
|356            |F7_P |
|382            |F8   |
|383            |F8_H |
|385            |F8_P |
|410            |F9   |
|411            |F9_H |
|413            |F9_P |
|424            |F10  |
|425            |F10_H|
|427            |F10_P|
|438            |F11  |
|439            |F11_H|
|441            |F11_P|
|454            |F12  |
|455            |F12_H|
|457            |F12_P|
|465            |F13  |
|466            |F13_H|
|468            

In [0]:
from pyspark.sql import functions as F

rows_with_index = (
    df_mapping_land_raw
    .withColumn(
        "row_array",
        F.array(*[F.col(c) for c in df_mapping_land_raw.columns])
    )
    .select(
        F.monotonically_increasing_id().alias("row_id"),
        "row_array"
    )
)

rows_with_index.show(15, truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
from pyspark.sql import functions as F

field_row = (
    df_mapping_land_raw
    .filter(
        F.array_contains(
            F.array(*[F.col(c) for c in df_mapping_land_raw.columns]),
            "F1"
        )
    )
    .first()
)

description_row = (
    df_mapping_land_raw
    .filter(
        F.array_contains(
            F.array(*[F.col(c) for c in df_mapping_land_raw.columns]),
            "GESAMTSTIMMEN (SPD)"
        )
    )
    .first()
)

In [0]:
print(field_row is not None)
print(description_row is not None)

True
False


In [0]:
description_row = (
    df_mapping_land_raw
    .filter(
        F.concat_ws(
            " ",
            *[F.col(c) for c in df_mapping_land_raw.columns]
        ).contains("GESAMTSTIMMEN (SPD)")
    )
    .first()
)

print(description_row is not None)

False


In [0]:
description_row = (
    df_mapping_land_raw
    .filter(
        F.lower(
            F.concat_ws(
                " ",
                *[F.col(c) for c in df_mapping_land_raw.columns]
            )
        ).contains("gesamtstimmen")
    )
    .first()
)

print(description_row is not None)

False


In [0]:
df_mapping_land_raw.select(
    "_c19",
    "_c20",
    "_c21",
    "_c22"
).show(20, truncate=False)

+------------------+---------------------+-------------------+---------------------+
|_c19              |_c20                 |_c21               |_c22                 |
+------------------+---------------------+-------------------+---------------------+
|NULL              |NULL                 |NULL               |NULL                 |
|NULL              |NULL                 |NULL               |NULL                 |
|NULL              |NULL                 |NULL               |NULL                 |
|NULL              |NULL                 |NULL               |NULL                 |
|NULL              |NULL                 |NULL               |NULL                 |
|F1                |F1_H                 |F1_0               |F1_P                 |
|NULL              |NULL                 |NULL               |NULL                 |
|GESAMSTIMMEN (SPD)|HEILUNGSSTIMMEN (SPD)|LISTENSTIMMEN (SPD)|PERSONENSTIMMEN (SPD)|
+------------------+---------------------+-------------------+---

In [0]:
rows = df_mapping_land_raw.collect()

field_row = rows[5]
description_row = rows[7]

In [0]:
import re

mapping_data = []

for col_name in df_mapping_land_raw.columns:
    field = field_row[col_name]
    description = description_row[col_name]

    if field is None or description is None:
        continue

    field = str(field).strip()
    description = str(description).strip()

    match = re.match(
        r"^(GESAMTSTIMMEN|HEILUNGSSTIMMEN|LISTENSTIMMEN|PERSONENSTIMMEN) \((.+)\)$",
        description
    )

    if field.startswith("F") and match:
        stimmenart = match.group(1)
        partei = match.group(2)

        mapping_data.append(
            (field, partei, stimmenart)
        )

In [0]:
land_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "bronze/state_elections/state=hamburg/"
    "ergebnis-download-land2025.csv"
)

df_land_2025 = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(land_path)
)

print("Rows:", df_land_2025.count())
print("Columns:", len(df_land_2025.columns))

df_land_2025.printSchema()
print(df_land_2025.columns[:40])

Rows: 1972
Columns: 512
root
 |-- Bezirk: string (nullable = true)
 |-- Wahlkreis: string (nullable = true)
 |-- Stadtteil: string (nullable = true)
 |-- Erfassungsgebietsnummer: integer (nullable = true)
 |-- Erfassungsgebietsart: string (nullable = true)
 |-- Ausgezaehlt abweichend Gebiet: string (nullable = true)
 |-- Ausgezaehlt abweichend AGS: string (nullable = true)
 |-- Ausbleibend: string (nullable = true)
 |-- Wahlberechtigte gesamt (A): integer (nullable = true)
 |-- Wahlberechtigte ohne Wahlschein (A1): integer (nullable = true)
 |-- Wahlberechtigte mit Wahlschein (A2): integer (nullable = true)
 |-- Wahlberechtigte nicht im WVZ (A3): integer (nullable = true)
 |-- Waehler gesamt (B): integer (nullable = true)
 |-- Waehler mit Wahlschein (B1): integer (nullable = true)
 |-- Waehler ohne Wahlschein (B2): integer (nullable = true)
 |-- Stimmzettel gesamt (B4): integer (nullable = true)
 |-- Stimmzettel ungueltig (E1): integer (nullable = true)
 |-- Stimmzettel gueltig (E2): i

In [0]:
import re

# Mapping-Datei ist schon eingelesen als:
# df_mapping_land_raw

rows = df_mapping_land_raw.collect()

# Zeile mit F1, F1_H, F1_0, F1_P ...
field_row = rows[5]

# Zeile mit GESAMTSTIMMEN (SPD), ...
description_row = rows[7]

mapping_data = []

for col_name in df_mapping_land_raw.columns:
    
    field = field_row[col_name]
    description = description_row[col_name]

    if field is None or description is None:
        continue

    field = str(field).strip()
    description = str(description).strip()

    match = re.match(
        r"^(GESAMTSTIMMEN|HEILUNGSSTIMMEN|LISTENSTIMMEN|PERSONENSTIMMEN) \((.+)\)$",
        description
    )

    if field.startswith("F") and match:
        stimmenart = match.group(1)
        partei = match.group(2)

        mapping_data.append(
            (field, partei, stimmenart)
        )

df_mapping = spark.createDataFrame(
    mapping_data,
    ["feld", "partei", "stimmenart"]
)

df_mapping.show(100, truncate=False)

+-----+-------------------+---------------+
|feld |partei             |stimmenart     |
+-----+-------------------+---------------+
|F1_H |SPD                |HEILUNGSSTIMMEN|
|F1_0 |SPD                |LISTENSTIMMEN  |
|F1_P |SPD                |PERSONENSTIMMEN|
|F2_H |CDU                |HEILUNGSSTIMMEN|
|F2_0 |CDU                |LISTENSTIMMEN  |
|F2_P |CDU                |PERSONENSTIMMEN|
|F3_H |FDP                |HEILUNGSSTIMMEN|
|F3_0 |FDP                |LISTENSTIMMEN  |
|F3_P |FDP                |PERSONENSTIMMEN|
|F4_H |GRÜNE              |HEILUNGSSTIMMEN|
|F4_0 |GRÜNE              |LISTENSTIMMEN  |
|F4_P |GRÜNE              |PERSONENSTIMMEN|
|F5_H |Volt               |HEILUNGSSTIMMEN|
|F5_0 |Volt               |LISTENSTIMMEN  |
|F5_P |Volt               |PERSONENSTIMMEN|
|F6_H |Die Linke          |HEILUNGSSTIMMEN|
|F6_0 |Die Linke          |LISTENSTIMMEN  |
|F6_P |Die Linke          |PERSONENSTIMMEN|
|F7_H |AfD                |HEILUNGSSTIMMEN|
|F7_0 |AfD                |LISTE

In [0]:
id_cols = [
    "Bezirk",
    "Wahlkreis",
    "Stadtteil",
    "Erfassungsgebietsnummer",
    "Erfassungsgebietsart",
    "Wahlberechtigte gesamt (A)",
    "Wahlberechtigte ohne Wahlschein (A1)",
    "Wahlberechtigte mit Wahlschein (A2)",
    "Wahlberechtigte nicht im WVZ (A3)",
    "Waehler gesamt (B)",
    "Waehler mit Wahlschein (B1)",
    "Waehler ohne Wahlschein (B2)",
    "Stimmzettel gesamt (B4)",
    "Stimmzettel ungueltig (E1)",
    "Stimmzettel gueltig (E2)",
    "Stimmen gueltige (F)"
]
mapping_rows = df_mapping.collect()

long_dfs = []

for row in mapping_rows:

    feld = row["feld"]
    partei = row["partei"]
    stimmenart = row["stimmenart"]

    temp = (
        df_land_2025
        .select(
            *id_cols,
            F.lit(partei).alias("partei"),
            F.lit(stimmenart).alias("stimmenart"),
            F.col(feld).alias("stimmen")
        )
    )

    long_dfs.append(temp)

In [0]:
df_long_2025 = long_dfs[0]

for temp in long_dfs[1:]:
    df_long_2025 = df_long_2025.unionByName(temp)

In [0]:
df_long_2025 = (
    df_long_2025
    .withColumn("wahljahr", F.lit(2025))
    .withColumn("wahltyp", F.lit("Bürgerschaftswahl"))
    .withColumn("bundesland", F.lit("Hamburg"))
)

In [0]:
df_long_2025.select(
    "Erfassungsgebietsnummer",
    "partei",
    "stimmenart",
    "stimmen"
).show(40, truncate=False)

+-----------------------+------+---------------+-------+
|Erfassungsgebietsnummer|partei|stimmenart     |stimmen|
+-----------------------+------+---------------+-------+
|10101                  |SPD   |HEILUNGSSTIMMEN|5      |
|10201                  |SPD   |HEILUNGSSTIMMEN|0      |
|14201                  |SPD   |HEILUNGSSTIMMEN|0      |
|15001                  |SPD   |HEILUNGSSTIMMEN|0      |
|1019901                |SPD   |HEILUNGSSTIMMEN|0      |
|10301                  |SPD   |HEILUNGSSTIMMEN|0      |
|10302                  |SPD   |HEILUNGSSTIMMEN|0      |
|10401                  |SPD   |HEILUNGSSTIMMEN|5      |
|10402                  |SPD   |HEILUNGSSTIMMEN|10     |
|10403                  |SPD   |HEILUNGSSTIMMEN|10     |
|1039901                |SPD   |HEILUNGSSTIMMEN|0      |
|1039902                |SPD   |HEILUNGSSTIMMEN|5      |
|1039903                |SPD   |HEILUNGSSTIMMEN|0      |
|10501                  |SPD   |HEILUNGSSTIMMEN|0      |
|10502                  |SPD   

In [0]:
df_long_2025.groupBy(
    "stimmenart"
).count().show()

+---------------+-----+
|     stimmenart|count|
+---------------+-----+
|HEILUNGSSTIMMEN|31552|
|  LISTENSTIMMEN|31552|
|PERSONENSTIMMEN|31552|
+---------------+-----+



In [0]:
df_long_2025.select("partei").distinct().show(50, truncate=False)

+-------------------+
|partei             |
+-------------------+
|SPD                |
|CDU                |
|FDP                |
|GRÜNE              |
|Volt               |
|Die Linke          |
|AfD                |
|DieWahl - WFG      |
|DAVA-Hamburg       |
|FREIE WÄHLER       |
|Die PARTEI         |
|ÖDP                |
|Tierschutzpartei   |
|BÜNDNIS DEUTSCHLAND|
|BSW                |
|NPD                |
+-------------------+



In [0]:
clean_cols = []

for c in df_long_2025.columns:
    c = (
        c.lower()
        .replace(" ", "_")
        .replace("ä", "ae")
        .replace("ö", "oe")
        .replace("ü", "ue")
        .replace("ß", "ss")
        .replace("(", "")
        .replace(")", "")
    )

    clean_cols.append(c)

df_long_2025 = df_long_2025.toDF(*clean_cols)

In [0]:
from pyspark.sql import functions as F

# 1. NULL-Werte prüfen
df_long_2025.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_long_2025.columns
]).show(truncate=False)

+------+---------+---------+-----------------------+--------------------+------------------------+----------------------------------+---------------------------------+-------------------------------+----------------+-------------------------+--------------------------+---------------------+------------------------+----------------------+------------------+------+----------+-------+--------+-------+----------+
|bezirk|wahlkreis|stadtteil|erfassungsgebietsnummer|erfassungsgebietsart|wahlberechtigte_gesamt_a|wahlberechtigte_ohne_wahlschein_a1|wahlberechtigte_mit_wahlschein_a2|wahlberechtigte_nicht_im_wvz_a3|waehler_gesamt_b|waehler_mit_wahlschein_b1|waehler_ohne_wahlschein_b2|stimmzettel_gesamt_b4|stimmzettel_ungueltig_e1|stimmzettel_gueltig_e2|stimmen_gueltige_f|partei|stimmenart|stimmen|wahljahr|wahltyp|bundesland|
+------+---------+---------+-----------------------+--------------------+------------------------+----------------------------------+---------------------------------+-------

In [0]:
rows = df_long_2025.count()
distinct_rows = df_long_2025.distinct().count()

print("Rows:", rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicates:", rows - distinct_rows)

Rows: 94656
Distinct rows: 94656
Exact duplicates: 0


In [0]:
df_long_2025.groupBy(
    "stimmenart"
).count().show()

df_long_2025.select(
    "partei"
).distinct().orderBy("partei").show(50, truncate=False)

+---------------+-----+
|     stimmenart|count|
+---------------+-----+
|  LISTENSTIMMEN|31552|
|PERSONENSTIMMEN|31552|
|HEILUNGSSTIMMEN|31552|
+---------------+-----+

+-------------------+
|partei             |
+-------------------+
|AfD                |
|BSW                |
|BÜNDNIS DEUTSCHLAND|
|CDU                |
|DAVA-Hamburg       |
|Die Linke          |
|Die PARTEI         |
|DieWahl - WFG      |
|FDP                |
|FREIE WÄHLER       |
|GRÜNE              |
|NPD                |
|SPD                |
|Tierschutzpartei   |
|Volt               |
|ÖDP                |
+-------------------+



In [0]:
df_long_2025.printSchema()

root
 |-- bezirk: string (nullable = true)
 |-- wahlkreis: string (nullable = true)
 |-- stadtteil: string (nullable = true)
 |-- erfassungsgebietsnummer: integer (nullable = true)
 |-- erfassungsgebietsart: string (nullable = true)
 |-- wahlberechtigte_gesamt_a: integer (nullable = true)
 |-- wahlberechtigte_ohne_wahlschein_a1: integer (nullable = true)
 |-- wahlberechtigte_mit_wahlschein_a2: integer (nullable = true)
 |-- wahlberechtigte_nicht_im_wvz_a3: integer (nullable = true)
 |-- waehler_gesamt_b: integer (nullable = true)
 |-- waehler_mit_wahlschein_b1: integer (nullable = true)
 |-- waehler_ohne_wahlschein_b2: integer (nullable = true)
 |-- stimmzettel_gesamt_b4: integer (nullable = true)
 |-- stimmzettel_ungueltig_e1: integer (nullable = true)
 |-- stimmzettel_gueltig_e2: integer (nullable = true)
 |-- stimmen_gueltige_f: integer (nullable = true)
 |-- partei: string (nullable = false)
 |-- stimmenart: string (nullable = false)
 |-- stimmen: integer (nullable = true)
 |-- wah

In [0]:
silver_path_2025 = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "silver/state_elections/state=hamburg/election_year=2025/"
)

df_long_2025.write.mode("overwrite").parquet(silver_path_2025)